# Brain v0.3 — Training Notebook

This notebook trains **Brain v0.3** (~33M, 4k context, NTK RoPE) and prepares
an **SFT / reasoning** stage with domain specialization.

**Domains supported via templates + data mixture:**
- NLP · Reasoning & Agentic AI · Legal & Cybersecurity
- Software Engineering · Finance (BFSI) · Advanced Text & Linguistics
- Logical & Mathematical Reasoning · Agentic Planning
- Code Architecture & Generation · Gen AI

**Speed:** SDPA already uses FlashAttention / mem-efficient kernels when available.
Optional helper: `brain.inference.speed.prefer_flash_attention`.

**Pipeline:**
1. Pretrain (this notebook, long context)
2. SFT on instruction + CoT + domain data (stage 2 cells)
3. (Later) Preference / DPO

## 0. GPU check

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > GPU")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory/1e9:.1f} GB")
    cap_major, _ = torch.cuda.get_device_capability(0)
    USE_BF16 = cap_major >= 8
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    print(f"Compute capability: {cap_major}.x -> using {'bf16' if USE_BF16 else 'fp16 (+ GradScaler)'}")
else:
    USE_BF16 = False
    AMP_DTYPE = torch.float16

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/brain-v0.3'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/sft', exist_ok=True)
print("Project dir:", PROJECT_DIR)

## 2. Load Brain v0.3 package

In [ ]:
import zipfile, glob, sys, pathlib

def find_repo_root(*search_dirs):
    for d in search_dirs:
        for req in pathlib.Path(d).rglob('requirements.txt'):
            # prefer v0.3 if multiple
            return req.parent
    return None

repo_root = find_repo_root(PROJECT_DIR, '/content')

if repo_root is None:
    zip_candidates = glob.glob(f'{PROJECT_DIR}/**/*.zip', recursive=True) + glob.glob('/content/*.zip')
    assert zip_candidates, (
        f"Upload brain_v0_3.zip to {PROJECT_DIR} or /content, then re-run."
    )
    zip_path = [p for p in zip_candidates if 'v0_3' in p or 'v0.3' in p]
    zip_path = zip_path[0] if zip_path else zip_candidates[0]
    print("Found zip:", zip_path)
    extract_to = '/content/repo_v03'
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_to)
    repo_root = find_repo_root(extract_to)
    assert repo_root is not None

print("Repo root:", repo_root)
sys.path.insert(0, str(repo_root))

In [ ]:
!pip install -q -r {repo_root}/requirements.txt 2>&1 | tail -5
!pip install -q datasets huggingface_hub tqdm

In [ ]:
import subprocess
result = subprocess.run(['python3', '-m', 'pytest', 'tests/', '-q'], cwd=str(repo_root),
                         env={**os.environ, 'PYTHONPATH': str(repo_root)}, capture_output=True, text=True)
print(result.stdout[-1500:] if result.stdout else "")
if result.returncode != 0:
    print(result.stderr[-1500:] if result.stderr else "")
    print("WARNING: some package tests failed — check before long training.")
else:
    print("Tests pass — safe to proceed.")

## 3. Config (v0.3)

- `CONTEXT_LEN` up to **4096** (model max). Start with 1024–2048 on T4 if VRAM is tight.
- `VOCAB_SIZE = 32768`
- Domain tags + CoT tokens are in the tokenizer special set for SFT later.

In [ ]:
TARGET_GB       = 3.0
CONTEXT_LEN     = 1024    # raise toward 2048/4096 when VRAM allows
BATCH_SIZE      = 16      # lower than v0.2 if using longer context
GRAD_ACCUM      = 8
LR              = 3e-4
MIN_LR          = 2e-5
WARMUP_STEPS    = 500
DROPOUT         = 0.0
CKPT_EVERY      = 500
GEN_EVERY       = 1000
VOCAB_SIZE      = 32768

CONFIG_PATH = repo_root / 'configs' / 'model' / 'brain_0_3_35m.yaml'
CKPT_DIR    = f'{PROJECT_DIR}/checkpoints'
DATA_DIR    = f'{PROJECT_DIR}/data'
TOKENIZER_PATH = f'{DATA_DIR}/brain_tokenizer_32k.json'
corpus_path = f'{DATA_DIR}/corpus_{TARGET_GB}gb.txt'
progress_path = f'{DATA_DIR}/corpus_{TARGET_GB}gb.progress.json'
token_bin_path = f'{DATA_DIR}/corpus_{TARGET_GB}gb.tokens.uint16'

MIX = [
    ("HuggingFaceFW/fineweb-edu", "sample-10BT", "train", "text", 0.40, False),
    ("bigcode/the-stack-smol",    "data/python",  "train", "content", 0.20, True),
    ("wikimedia/wikipedia",       "20231101.en",  "train", "text", 0.15, False),
    ("open-web-math/open-web-math", None,         "train", "text", 0.15, False),
    # light reasoning-oriented text if available; otherwise fineweb covers general
]

## 4. Build corpus (resumable) — same logic as v0.2 notebook

In [ ]:
import re, ast, json
from datasets import load_dataset

MIN_DOC_CHARS, MAX_SYMBOL_FRACTION = 800, 0.30
MIN_CODE_CHARS, MIN_COMMENT_RATIO = 600, 0.10
_WS_RUN = re.compile(r"[ \t]{3,}")
_REPEATED_LINE_RUN = re.compile(r"(?:^(.*)$\n)(?:\1\n){2,}", re.MULTILINE)

def passes_heuristic_filters(text):
    if len(text) < MIN_DOC_CHARS:
        return False
    symbol_chars = sum(1 for c in text if not (c.isalnum() or c.isspace()))
    return symbol_chars / max(len(text), 1) <= MAX_SYMBOL_FRACTION

def clean_text(text):
    text = _WS_RUN.sub(" ", text)
    text = _REPEATED_LINE_RUN.sub(lambda m: m.group(1) + "\n", text)
    return text.strip()

def passes_code_filters(code_text):
    if len(code_text) < MIN_CODE_CHARS:
        return False
    try:
        ast.parse(code_text)
    except (SyntaxError, ValueError):
        return False
    lines = [l for l in code_text.splitlines() if l.strip()]
    if not lines:
        return False
    comment_lines = sum(1 for l in lines if l.strip().startswith("#"))
    triple1, triple2 = chr(34)*3, chr(39)*3
    comment_lines += code_text.count(triple1) // 2 + code_text.count(triple2) // 2
    return comment_lines / len(lines) >= MIN_COMMENT_RATIO

TARGET_BYTES = int(TARGET_GB * 1e9)

if os.path.exists(corpus_path) and os.path.getsize(corpus_path) >= TARGET_BYTES * 0.98:
    print(f"Corpus already complete at {corpus_path} ({os.path.getsize(corpus_path)/1e9:.2f} GB) — skipping.")
else:
    progress = json.load(open(progress_path)) if os.path.exists(progress_path) else {}
    mode = 'a' if progress else 'w'
    with open(corpus_path, mode, encoding='utf-8') as out:
        for name, subset, split, field, weight, is_code in MIX:
            budget = int(TARGET_BYTES * weight)
            state = progress.get(name, {"bytes": 0, "docs_consumed": 0})
            if state["bytes"] >= budget:
                print(f"{name}: already complete — skipping")
                continue
            print(f"Streaming {name} -> budget {budget/1e9:.2f} GB")
            try:
                ds = load_dataset(name, subset, split=split, streaming=True) if subset else \
                     load_dataset(name, split=split, streaming=True)
            except Exception as e:
                print(f"  SKIPPED ({name}): {e}")
                continue
            if state["docs_consumed"]:
                ds = ds.skip(state["docs_consumed"])
            for i, row in enumerate(ds):
                text = (row.get(field) or "").strip()
                state["docs_consumed"] += 1
                if not text:
                    continue
                ok = passes_code_filters(text) if is_code else passes_heuristic_filters(text)
                if not ok:
                    continue
                if not is_code:
                    text = clean_text(text)
                out.write(text); out.write("\n\n")
                state["bytes"] += len(text.encode('utf-8'))
                if i % 200 == 0:
                    out.flush(); os.fsync(out.fileno())
                    progress[name] = state
                    json.dump(progress, open(progress_path, 'w'))
                if state["bytes"] >= budget:
                    break
            progress[name] = state
            json.dump(progress, open(progress_path, 'w'))
            print(f"  {name}: {state['bytes']/1e9:.2f} GB saved")
    print(f"Total corpus: {os.path.getsize(corpus_path)/1e9:.2f} GB")

## 5. Train tokenizer (32k + special tokens for chat/CoT/domains)

In [ ]:
from brain.tokenizer.tokenizer import BrainTokenizer
from brain.tokenizer.special_tokens import ALL_SPECIAL_TOKENS

print("Special tokens registered:", len(ALL_SPECIAL_TOKENS))
print(ALL_SPECIAL_TOKENS[:12], "...")

if os.path.exists(TOKENIZER_PATH):
    tok = BrainTokenizer.from_file(TOKENIZER_PATH)
    print(f"Loaded existing tokenizer, vocab_size={tok.vocab_size}")
else:
    TOKENIZER_TRAIN_SAMPLE_MB = 300
    sample_path = f'{DATA_DIR}/tokenizer_train_sample.txt'
    with open(corpus_path, 'r', encoding='utf-8') as src, open(sample_path, 'w', encoding='utf-8') as dst:
        dst.write(src.read(TOKENIZER_TRAIN_SAMPLE_MB * 1_000_000))
    tok = BrainTokenizer.train(corpus_paths=sample_path, vocab_size=VOCAB_SIZE, save_path=TOKENIZER_PATH)
    print(f"Trained tokenizer, vocab_size={tok.vocab_size}")

## 6. Tokenize corpus → memmap

In [ ]:
import numpy as np

if os.path.exists(token_bin_path):
    n_tokens = os.path.getsize(token_bin_path) // 2
    print(f"Token file exists: {n_tokens:,} tokens")
else:
    CHUNK_CHARS = 20_000_000
    total_tokens = 0
    with open(corpus_path, 'r', encoding='utf-8') as f, open(token_bin_path, 'wb') as out:
        while True:
            chunk = f.read(CHUNK_CHARS)
            if not chunk:
                break
            ids = tok.encode(chunk, add_special_tokens=False)
            arr = np.array(ids, dtype=np.uint16)
            out.write(arr.tobytes())
            total_tokens += len(ids)
    print(f"Tokenized {total_tokens:,} tokens")

n_tokens = os.path.getsize(token_bin_path) // 2
tokens = np.memmap(token_bin_path, dtype=np.uint16, mode='r', shape=(n_tokens,))
print(f"{n_tokens:,} tokens memmapped")

## 7. Build model (v0.3 config)

In [ ]:
from brain.core.config import BrainConfig
from brain.model.brain_model import BrainForCausalLM
from brain.inference.speed import sdpa_backends_available, prefer_flash_attention

print("SDPA backends:", sdpa_backends_available())

cfg = BrainConfig.from_yaml(CONFIG_PATH)
if DROPOUT > 0:
    cfg.dropout = DROPOUT

model = BrainForCausalLM(cfg).cuda()
counts = model.count_parameters()
print(f"Parameters: {counts['total']:,}  (~{counts['total']/1e6:.1f}M)")
print(f"max_position_embeddings={cfg.max_position_embeddings}, rope_theta={cfg.rope_theta}")
print(f"rope_scaling={cfg.rope_scaling_type} x{cfg.rope_scaling_factor}")

optimal_tokens = counts['total'] * 20
print(f"Chinchilla-optimal ~{optimal_tokens/1e9:.3f}B tokens")
print(f"Corpus tokens: {n_tokens/1e9:.3f}B")

EPOCHS = 2
tokens_per_step = BATCH_SIZE * GRAD_ACCUM * CONTEXT_LEN
MAX_STEPS = max(1, int((n_tokens * EPOCHS) / tokens_per_step))
print(f"EPOCHS={EPOCHS} -> MAX_STEPS={MAX_STEPS:,}")

## 8. Pretrain loop

In [ ]:
import math, time, csv

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR * step / max(1, WARMUP_STEPS)
    progress_frac = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return MIN_LR + 0.5 * (LR - MIN_LR) * (1 + math.cos(math.pi * min(progress_frac, 1.0)))

def get_batch(tokens_array, lo, hi, ctx_len, batch_size):
    max_start = hi - ctx_len - 1
    starts = np.random.randint(lo, max_start, size=batch_size)
    x = np.stack([tokens_array[s:s+ctx_len] for s in starts]).astype(np.int64)
    return torch.from_numpy(x).cuda()

@torch.no_grad()
def estimate_val_loss(model_obj, tokens_array, lo, hi, ctx_len, batch_size, iters=8):
    model_obj.eval()
    losses = []
    for _ in range(iters):
        x = get_batch(tokens_array, lo, hi, ctx_len, batch_size)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            out = model_obj(x, labels=x)
        losses.append(out['loss'].item())
    model_obj.train()
    return sum(losses) / len(losses)

train_cutoff = int(n_tokens * 0.98)
print(f"Train: [0, {train_cutoff:,})   Val: [{train_cutoff:,}, {n_tokens:,})")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=not USE_BF16)
metrics_path = f'{PROJECT_DIR}/training_metrics.csv'

start_step = 0
resume_ckpts = sorted(glob.glob(f'{CKPT_DIR}/resume_step_*.pt'))
if resume_ckpts:
    latest = resume_ckpts[-1]
    print(f"Resuming from {latest}")
    payload = torch.load(latest, map_location='cuda')
    model.load_state_dict(payload['model_state_dict'])
    optimizer.load_state_dict(payload['optimizer_state_dict'])
    start_step = payload['step'] + 1
else:
    print("No checkpoint found — starting fresh.")
    with open(metrics_path, 'w', newline='') as f:
        csv.writer(f).writerow(['step', 'train_loss', 'val_loss', 'lr', 'tokens_per_sec'])

model.train()
t0 = time.time()
for step in range(start_step, MAX_STEPS):
    for g in optimizer.param_groups:
        g['lr'] = lr_at(step)

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for _ in range(GRAD_ACCUM):
        x = get_batch(tokens, 0, train_cutoff, CONTEXT_LEN, BATCH_SIZE)
        with prefer_flash_attention():
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                out = model(x, labels=x)
                loss = out['loss'] / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if step % 50 == 0:
        toks_per_sec = (step - start_step + 1) * BATCH_SIZE * GRAD_ACCUM * CONTEXT_LEN / max(time.time() - t0, 1e-6)
        val_loss = estimate_val_loss(model, tokens, train_cutoff, n_tokens, CONTEXT_LEN, BATCH_SIZE)
        print(f"step {step:>6}  train {accum_loss:.4f}  val {val_loss:.4f}  lr {lr_at(step):.2e}  {toks_per_sec:,.0f} tok/s")
        with open(metrics_path, 'a', newline='') as f:
            csv.writer(f).writerow([step, round(accum_loss, 4), round(val_loss, 4), f"{lr_at(step):.2e}", int(toks_per_sec)])

    if step % CKPT_EVERY == 0 and step > start_step:
        resume_path = f'{CKPT_DIR}/resume_step_{step:07d}.pt'
        torch.save({'step': step, 'model_state_dict': model.state_dict(),
                     'optimizer_state_dict': optimizer.state_dict()}, resume_path)
        for old in sorted(glob.glob(f'{CKPT_DIR}/resume_step_*.pt'))[:-3]:
            os.remove(old)
        model.save_checkpoint(f'{CKPT_DIR}/brain_step_{step:07d}.pt',
                               tokenizer_path=TOKENIZER_PATH,
                               training_metadata={'step': step, 'loss': accum_loss})
        print(f"  saved checkpoints at step {step}")

    if step % GEN_EVERY == 0 and step > start_step:
        model.eval()
        prompt_ids = torch.tensor([tok.encode("The", add_special_tokens=False)], device='cuda')
        with prefer_flash_attention():
            out = model.generate(prompt_ids, max_new_tokens=40, temperature=0.8, top_k=40)
        print("  sample:", tok.decode(out[0].tolist()))
        model.train()

print("Pretrain done (or interrupted — re-run to resume).")

## 9. SFT stage — instruction + CoT + domains

After pretraining, run a shorter SFT phase on instruction data.

**Domains** (use `format_domain_prompt` / `format_cot`):
NLP, reasoning, legal, cyber, software, finance, linguistics, math, agent, code, genai

Example: build a small JSONL of `{"domain","user","assistant","use_cot"}` then pack with the templates below.

In [ ]:
from brain.sft.templates import format_chat, format_cot, format_domain_prompt, DOMAIN_SYSTEM_PROMPTS
from brain.sft.domains import DOMAINS, list_domains

print("Domains:", list_domains())
print()
# Demo formatting
print("--- math CoT example ---")
print(format_cot(
    "What is 17 * 19?",
    domain="math",
    steps=["17*20=340", "subtract 17 -> 323"],
    answer="323",
))
print()
print("--- code domain ---")
print(format_domain_prompt("Write a binary search in Python.", domain="code", use_cot=False)[:300], "...")

In [ ]:
# Minimal SFT loop sketch (fill SFT_JSONL with your data)
# Each line: {"domain": "math", "user": "...", "assistant": "...", "use_cot": true}

SFT_JSONL = f'{PROJECT_DIR}/sft/instructions.jsonl'
SFT_LR = 1e-4
SFT_STEPS = 2000
SFT_CTX = min(CONTEXT_LEN, 2048)

def load_sft_examples(path, limit=None):
    rows = []
    if not os.path.exists(path):
        print(f"No SFT file at {path} — create one to run stage 2.")
        return rows
    with open(path) as f:
        for i, line in enumerate(f):
            rows.append(json.loads(line))
            if limit and i + 1 >= limit:
                break
    return rows

sft_rows = load_sft_examples(SFT_JSONL)
print(f"SFT examples loaded: {len(sft_rows)}")

if sft_rows:
    # simple random batch from formatted strings
    def sft_batch(batch_size=4):
        import random
        exs = random.sample(sft_rows, min(batch_size, len(sft_rows)))
        texts = []
        for ex in exs:
            if ex.get("use_cot"):
                t = format_cot(ex["user"], domain=ex.get("domain", "reasoning"),
                               answer=ex.get("assistant"), reasoning=ex.get("reasoning"))
            else:
                t = format_chat(ex["user"], domain=ex.get("domain"), assistant=ex.get("assistant"))
            texts.append(t)
        # pad/truncate
        ids = [tok.encode(t, add_special_tokens=True)[:SFT_CTX] for t in texts]
        max_l = max(len(x) for x in ids)
        pad_id = 0
        arr = np.full((len(ids), max_l), pad_id, dtype=np.int64)
        for i, row in enumerate(ids):
            arr[i, :len(row)] = row
        return torch.from_numpy(arr).cuda()

    model.train()
    opt_sft = torch.optim.AdamW(model.parameters(), lr=SFT_LR, weight_decay=0.01)
    for step in range(SFT_STEPS):
        x = sft_batch(BATCH_SIZE)
        with prefer_flash_attention():
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                out = model(x, labels=x)
                loss = out['loss']
        opt_sft.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt_sft)
        scaler.update()
        if step % 50 == 0:
            print(f"sft step {step}  loss {loss.item():.4f}")
    model.save_checkpoint(f'{CKPT_DIR}/brain_sft_final.pt', tokenizer_path=TOKENIZER_PATH)
    print("SFT stage complete.")
else:
    print("Skip SFT until instructions.jsonl exists.")

## 10. Playground (domain-aware)

In [ ]:
from brain.sft.templates import format_domain_prompt

resume_ckpts = sorted(glob.glob(f'{CKPT_DIR}/resume_step_*.pt') + glob.glob(f'{CKPT_DIR}/brain_sft*.pt'))
if not resume_ckpts:
    print("No checkpoints yet.")
else:
    path = resume_ckpts[-1]
    if path.endswith('sft_final.pt') or 'brain_sft' in path:
        # full checkpoint API
        try:
            model = BrainForCausalLM.load_checkpoint(path).cuda()
        except Exception:
            payload = torch.load(path, map_location='cuda')
            model.load_state_dict(payload.get('model_state_dict', payload))
    else:
        payload = torch.load(path, map_location='cuda')
        model.load_state_dict(payload['model_state_dict'])
    model.eval()
    print("Loaded", path)

    tests = [
        ("math", "What is the derivative of x^3?"),
        ("code", "Write a Python function to reverse a linked list."),
        ("finance", "What is the difference between equity and debt financing?"),
        ("cyber", "List three basic defenses against phishing."),
    ]
    with torch.no_grad():
        for domain, q in tests:
            prompt = format_domain_prompt(q, domain=domain, use_cot=(domain in ("math", "reasoning")))
            ids = torch.tensor([tok.encode(prompt, add_special_tokens=False)], device='cuda')
            with prefer_flash_attention():
                out = model.generate(ids, max_new_tokens=80, temperature=0.7, top_k=40)
            print(f"\n=== {domain} ===\n{tok.decode(out[0].tolist())}")

## 11. Plot loss

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = f'{PROJECT_DIR}/training_metrics.csv'
if not os.path.exists(metrics_path):
    print("No metrics yet.")
else:
    df = pd.read_csv(metrics_path)
    if len(df) < 2:
        print("Not enough rows.")
    else:
        plt.figure(figsize=(10, 4.5))
        plt.plot(df['step'], df['train_loss'], label='train')
        plt.plot(df['step'], df['val_loss'], label='val', linestyle='--')
        plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
        plt.title('Brain v0.3 pretrain')
        plt.tight_layout(); plt.show()